In [3]:
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt
from breakage_population_balance import breakageModel

from test_fitting import x, zero, extract_frac_psm
from test_fitting_UVvis import interp

%matplotlib inline

In [4]:
# Using the method of classes solution with fraction, not that it's needed here.
class MoC_Fraction:
    def __init__(self, F0, t, x, phi, k):
        self.no_of_classes = F0.shape[0] # Number of classes
        self.phi = phi  # set the kernel
        self.k = k(x) # Calulate the frequency at size x
        self.x = x # require uniformly spaced grid
        # Solve with odeint, could use other implementation.
        self.F = odeint(lambda FF, t: self.RHS(FF, t), F0, t)
    
    def RHS(self, F, t): # Calculates dF/dt
        dFdt = np.zeros_like(F) # Initialise temporary array
        dFdt += np.nansum(self.phi*self.k*F, axis=-1) # Birth breakup term
        dFdt -= F * self.k # Death breakup term
        return dFdt

t = np.linspace(0, 3000, 600+1, dtype=np.float64)

In [5]:
S = 1e-6

key = "1500"

PSM, y_fracs = extract_frac_psm(key)

def run_simulation(array):
    b=array[0]
    e=array[1]
    a=array[2]
    s=array[3]
    # rate function
    k = lambda x: (x/(b*1e-3))**e

    # kernel function
    def Phi(x, y):
        if x < y:
            # Spherical chickens in a vacuum
            v_ = (1/6) * np.pi * y ** 3
            v  = (1/6) * np.pi * x ** 3
            return ( v / v_ )**(a*1e-1)
        else:
            return 0
    Phi = np.vectorize(Phi, otypes=[np.float64])

    # Run and solve the breakage problem
    model = breakageModel(zero, t, x, k, kernel=Phi)
    solution = model.solve("pyf_simple_breakage")

    return solution

def error(array):
    solution = run_simulation(array)

    err_sum = 0

    for time, psd in PSM:
        sol = solution[time_to_step(time)]
        err = np.sum(np.abs(psd-sol))
        err_sum += err
    
    UVvis_fit = interp(solution, section_value=S)
    
    np.sqrt()

    return np.array(err_sum)
